# 1 - Load Data Set

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import csv


In [2]:
# NOTE: Load only the training split

train = pd.read_csv("data/processed/train.csv")
print("Train shape:", train.shape)


Train shape: (48978, 41)


In [3]:
# Create output folder for saved EDA figures
Path("reports/figures").mkdir(parents=True, exist_ok=True)


# 2 - Check Numerical Categories

In [4]:
# Check for outliers
# Summarizes ranges, zeros, skew, and unusually large counts.
# No automatic outlier deletion is applied.

num_col = [
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient", "number_diagnoses",
]

# --- Summary table ---
summary = train[num_col].describe(percentiles=[0.01, 0.5, 0.99]).T[
    ["min", "1%", "50%", "99%", "max", "mean", "std"]
]

q1, q3 = train[num_col].quantile(0.25), train[num_col].quantile(0.75)
upper_fence = q3 + 1.5 * (q3 - q1)          # common rule of thumb for "unusually high"

summary["upper_fence"]    = upper_fence
summary["n_above_fence"]  = (train[num_col] > upper_fence).sum()
summary["pct_above"]      = (summary["n_above_fence"] / len(train) * 100).round(2)
summary["pct_zero"]       = ((train[num_col] == 0).mean() * 100).round(1)
summary["skew"]           = train[num_col].skew().round(2)

print(summary.round(2).to_string())


# time_in_hospital and num_lab_procedures, num_diagnoses > 0; leave as is
# no negative values
# number_outpatient, number_emergency, number_inpatient > 85% as 0 --> need to transform

# The utilization variables are highly right-skewed, with most observations equal to zero. 
# These zeros represent legitimate absence of prior utilization rather than missing or invalid values. 
# No observations will be removed as outliers. 
# If necessary, transformations such as log1p may be evaluated during model development, particularly for models sensitive to skewed numeric variables.


                    min   1%   50%   99%    max   mean    std  upper_fence  n_above_fence  pct_above  pct_zero   skew
time_in_hospital    1.0  1.0   3.0  13.0   14.0   4.27   2.93         12.0           1004       2.05       0.0   1.18
num_lab_procedures  1.0  1.0  44.0  85.0  132.0  42.96  19.90         96.0             70       0.14       0.0  -0.22
num_procedures      0.0  0.0   1.0   6.0    6.0   1.42   1.75          5.0           2664       5.44      44.2   1.23
num_medications     1.0  3.0  14.0  44.0   79.0  15.67   8.29         35.0           1309       2.67       0.0   1.44
number_outpatient   0.0  0.0   0.0   4.0   42.0   0.28   1.08          0.0           6390      13.05      87.0  10.21
number_emergency    0.0  0.0   0.0   2.0   25.0   0.10   0.47          0.0           3588       7.33      92.7  10.53
number_inpatient    0.0  0.0   0.0   3.0   11.0   0.18   0.60          0.0           5724      11.69      88.3   5.50
number_diagnoses    1.0  2.0   8.0   9.0   16.0   7.24  

In [5]:
# Plot the numerical variable distribution (histogram) and outliers (boxplots)
# Histograms show distribution shape, while boxplots highlight extreme values. 
# Skew suggests evaluating transformations for suitable models, rather than assuming every large value should be removed.

# Create 2 x 8 subplot layout
fig = make_subplots(
    rows=2,
    cols=len(num_col),
    subplot_titles=num_col,
    vertical_spacing=0.15,
    horizontal_spacing=0.03
)

for i, col in enumerate(num_col):
    
    # --- Top row: Histogram ---
    fig.add_trace(
        go.Histogram(
            x=train[col],
            nbinsx=30,
            showlegend=False
        ),
        row=1,
        col=i + 1
    )
    
    # --- Bottom row: Boxplot ---
    fig.add_trace(
        go.Box(
            y=train[col],
            boxpoints="outliers",
            showlegend=False
        ),
        row=2,
        col=i + 1
    )

# Axis labels
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Value", row=2, col=1)

# Overall layout
fig.update_layout(
    title="Numeric Variable Distributions and Outliers",
    template="plotly_white",
    height=650,
    width=1800,
    bargap=0.05
)

fig.show()


In [6]:
# Compare numeric variables by 30-day readmission status
# Differences are descriptive associations; they do not show that changing a variable would prevent readmission.
numeric_target_summary = []

for col in num_col:
    for target_value, label in [(0, "No 30-day readmission"),
                                (1, "30-day readmission")]:
        
        values = train.loc[train["readmit_30"] == target_value, col]
        
        numeric_target_summary.append({
            "Variable": col,
            "Readmission Status": label,
            "N": values.count(),
            "Mean": values.mean(),
            "Median": values.median(),
            "Q1": values.quantile(0.25),
            "Q3": values.quantile(0.75),
            "IQR": values.quantile(0.75) - values.quantile(0.25)
        })

numeric_target_summary = pd.DataFrame(numeric_target_summary)

print(numeric_target_summary.round(2).to_string(index=False))


          Variable    Readmission Status     N  Mean  Median   Q1   Q3  IQR
  time_in_hospital No 30-day readmission 44585  4.22     3.0  2.0  6.0  4.0
  time_in_hospital    30-day readmission  4393  4.79     4.0  2.0  6.0  4.0
num_lab_procedures No 30-day readmission 44585 42.77    44.0 31.0 57.0 26.0
num_lab_procedures    30-day readmission  4393 44.89    46.0 33.0 58.0 25.0
    num_procedures No 30-day readmission 44585  1.42     1.0  0.0  2.0  2.0
    num_procedures    30-day readmission  4393  1.42     1.0  0.0  2.0  2.0
   num_medications No 30-day readmission 44585 15.57    14.0 10.0 19.0  9.0
   num_medications    30-day readmission  4393 16.64    15.0 11.0 21.0 10.0
 number_outpatient No 30-day readmission 44585  0.28     0.0  0.0  0.0  0.0
 number_outpatient    30-day readmission  4393  0.31     0.0  0.0  0.0  0.0
  number_emergency No 30-day readmission 44585  0.10     0.0  0.0  0.0  0.0
  number_emergency    30-day readmission  4393  0.16     0.0  0.0  0.0  0.0
  number_inp

In [7]:
# Plot boxplot for numeric variables by 30-Day Readmission Status

fig = make_subplots(
    rows=4,
    cols=2,
    subplot_titles=num_col,
    vertical_spacing=0.10,
    horizontal_spacing=0.10
)

for i, col in enumerate(num_col):
    
    row = (i // 2) + 1
    col_num = (i % 2) + 1
    
    # No 30-day readmission
    fig.add_trace(
        go.Box(
            y=train.loc[train["readmit_30"] == 0, col],
            name="No",
            boxpoints="outliers",
            showlegend=(i == 0)
        ),
        row=row,
        col=col_num
    )
    
    # 30-day readmission
    fig.add_trace(
        go.Box(
            y=train.loc[train["readmit_30"] == 1, col],
            name="Yes",
            boxpoints="outliers",
            showlegend=(i == 0)
        ),
        row=row,
        col=col_num
    )

    fig.update_yaxes(
        title_text=col,
        row=row,
        col=col_num
    )

fig.update_layout(
    title="Numeric Variables by 30-Day Readmission Status",
    template="plotly_white",
    height=1200,
    width=1000
)

fig.show()


In [8]:
# Compare median values between the two target groups

median_comparison = (
    train
    .groupby("readmit_30")[num_col]
    .median()
    .T
    .rename(columns={
        0: "No 30-day readmission",
        1: "30-day readmission"
    })
)

median_comparison["Difference"] = (
    median_comparison["30-day readmission"]
    - median_comparison["No 30-day readmission"]
)

print(median_comparison.round(2).to_string())


# --- Create one figure with 8 panels ---

fig = make_subplots(
    rows=4,
    cols=2,
    subplot_titles=num_col,
    vertical_spacing=0.12,
    horizontal_spacing=0.10
)

for i, col in enumerate(num_col):
    row = (i // 2) + 1
    col_num = (i % 2) + 1

    fig.add_trace(
        go.Bar(
            x=["No 30-day readmission", "30-day readmission"],
            y=[
                median_comparison.loc[col, "No 30-day readmission"],
                median_comparison.loc[col, "30-day readmission"]
            ],
            showlegend=False
        ),
        row=row,
        col=col_num
    )

    fig.update_yaxes(
        title_text="Median",
        row=row,
        col=col_num
    )

fig.update_layout(
    title="Median Numeric Variables by 30-Day Readmission Status",
    height=1000,
    width=1000,
    template="plotly_white"
)

fig.show()


readmit_30          No 30-day readmission  30-day readmission  Difference
time_in_hospital                      3.0                 4.0         1.0
num_lab_procedures                   44.0                46.0         2.0
num_procedures                        1.0                 1.0         0.0
num_medications                      14.0                15.0         1.0
number_outpatient                     0.0                 0.0         0.0
number_emergency                      0.0                 0.0         0.0
number_inpatient                      0.0                 0.0         0.0
number_diagnoses                      8.0                 8.0         0.0


**Explanation / key finding:** Median comparisons emphasize typical values, but utilization medians can both be zero despite differences in the upper tail. Use them alongside means, quartiles, and distributions.

# 3 - Check Categorical Variables

In [9]:
# Define target variable

target = "readmit_30"

base_rate = train[target].mean() * 100

print(f"Overall 30-day readmission rate: {base_rate:.2f}%")


Overall 30-day readmission rate: 8.97%


**Explanation / key finding:** The training positive rate is about 8.97%. This is the reference rate used in categorical plots and the approximate constant-score average-precision baseline.

In [10]:
# ID columns as categorical

id_cols = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

for col in id_cols:
    train[col] = train[col].astype(str)


**Explanation / key finding:** Converts administrative IDs to strings so EDA treats them as categories. Their numeric labels do not represent a meaningful continuous scale.

In [11]:
# identify medication categorical columns

cat_cols = train.select_dtypes(exclude="number").columns.tolist()

print(f"Categorical columns: {len(cat_cols)}")


Categorical columns: 32


**Explanation / key finding:** Identifies categorical columns from their data types. This depends on the preceding ID conversion; verify the list before grouping or plotting.

In [12]:
# Identify medication columns:

med_values = {"No", "Steady", "Up", "Down"}

med_cols = [
    col for col in cat_cols
    if set(train[col].dropna().unique()).issubset(med_values)
]

print(f"Medication columns: {len(med_cols)}")
print(med_cols)


Medication columns: 17
['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin']


**Explanation / key finding:** Finds columns whose observed labels fit medication statuses. This is a convenience rule, not a clinical definition; verify the selected columns before interpreting medication counts.

In [13]:
# Create categorical readmission summary function

def rate_table(col):

    table = train.groupby(col)[target].agg(
        n="size",
        positives="sum",
        readmit_rate="mean"
    )

    table["pct_of_train"] = table["n"] / len(train) * 100
    table["readmit_rate_pct"] = table["readmit_rate"] * 100

    table = table[
        ["n", "pct_of_train", "positives", "readmit_rate_pct"]
    ]

    return table.sort_values("n", ascending=False).round(2)


**Explanation / key finding:** The helper reports category size, positive count, and readmission rate. Always consider sample size: a high rate in a small category may be unstable.

In [14]:
# Define key categorical variables:

key_cols = [
    "age",
    "race",
    "gender",
    "A1Cresult",
    "max_glu_serum",
    "change",
    "diabetesMed",
    "diag_1_group",
]

diag_cols = [
    "diag_1_group",
    "diag_2_group",
    "diag_3_group"
]
# Display readmission rate

for col in key_cols:
    print(f"\n=== {col} ===")
    print(rate_table(col).head(12).to_string())



=== age ===
              n  pct_of_train  positives  readmit_rate_pct
age                                                       
[70-80)   12391         25.30       1257             10.14
[60-70)   11021         22.50       1013              9.19
[50-60)    8672         17.71        605              6.98
[80-90)    7750         15.82        837             10.80
[40-50)    4772          9.74        358              7.50
[30-40)    1861          3.80        123              6.61
[90-100)   1256          2.56        123              9.79
[20-30)     799          1.63         58              7.26
[10-20)     360          0.74         17              4.72
[0-10)       96          0.20          2              2.08

=== race ===
                     n  pct_of_train  positives  readmit_rate_pct
race                                                             
Caucasian        36621         74.77       3360              9.18
AfricanAmerican   8874         18.12        768              8.65
U

**Explanation / key finding:** Displays rates for key demographic, lab, medication-change, and diagnosis variables. Similar individual rates do not prove a feature has no predictive value in combination with others.

In [15]:
# Wilson 95% confidence interval
def wilson_ci(positives, n, z=1.96):

    p = positives / n

    denominator = 1 + z**2 / n

    centre = (
        p + z**2 / (2 * n)
    ) / denominator

    margin = (
        z
        * np.sqrt(
            (p * (1 - p) / n)
            + (z**2 / (4 * n**2))
        )
        / denominator
    )

    return (
        (centre - margin) * 100,
        (centre + margin) * 100
    )


**Explanation / key finding:** Wilson intervals express uncertainty around an observed category proportion. Wider intervals usually accompany smaller groups; they are not uncertainty intervals for the model itself.

In [16]:
# Plot key columns

fig = make_subplots(
    rows=2,
    cols=4,
    subplot_titles=key_cols,
    vertical_spacing=0.2,
    horizontal_spacing=0.08
)

for i, col in enumerate(key_cols):

    # Get the 10 most common categories
    table = rate_table(col).head(10).reset_index()

    # Only show categories with more than 200 observations
    table = table[table["n"] > 200].copy()

    # Sort readmission rate from highest to lowest
    table = table.sort_values(
        "readmit_rate_pct",
        ascending=True
    ).reset_index(drop=True)

    # Calculate 95% Wilson confidence intervals
    ci = table.apply(
        lambda row: wilson_ci(
            row["positives"],
            row["n"]
        ),
        axis=1
    )

    table["ci_low"] = [x[0] for x in ci]
    table["ci_high"] = [x[1] for x in ci]

    # Calculate error bar lengths
    table["error_low"] = (
        table["readmit_rate_pct"] - table["ci_low"]
    )

    table["error_high"] = (
        table["ci_high"] - table["readmit_rate_pct"]
    )

    # Highlight categories above the overall base rate
    colors = [
        "#D55E00" if rate > base_rate else "#BDBDBD"
        for rate in table["readmit_rate_pct"]
    ]

    # Position in subplot
    row = (i // 4) + 1
    col_num = (i % 4) + 1

    # Add bar chart
    fig.add_trace(
        go.Bar(
            x=table[col].astype(str),
            y=table["readmit_rate_pct"],
            marker_color=colors,

            error_y=dict(
                type="data",
                symmetric=False,
                array=table["error_high"],
                arrayminus=table["error_low"],
                visible=True
            ),

            customdata=np.stack(
                (
                    table["n"],
                    table["ci_low"],
                    table["ci_high"]
                ),
                axis=-1
            ),

            hovertemplate=(
                "Category: %{x}<br>"
                "N: %{customdata[0]:,.0f}<br>"
                "Readmission rate: %{y:.1f}%<br>"
                "95% CI: %{customdata[1]:.1f}%–%{customdata[2]:.1f}%"
                "<extra></extra>"
            ),

            showlegend=False
        ),
        row=row,
        col=col_num
    )

    # Overall readmission base rate
    fig.add_hline(
        y=base_rate,
        line_dash="dash",
        line_color="black",
        line_width=1,
        row=row,
        col=col_num
    )

    # Axis formatting
    fig.update_yaxes(
        title_text="Readmission rate (%)",
        row=row,
        col=col_num
    )

    fig.update_xaxes(
        tickangle=-45,
        row=row,
        col=col_num
    )


fig.update_layout(
    title=(
        "30-Day Readmission Rate by Categorical Variables"
        f"<br><sup>Dashed line = overall training base rate ({base_rate:.1f}%) | "
        "Highlighted bars = above base rate | Error bars = 95% CI</sup>"
    ),
    template="plotly_white",
    height=900,
    width=1600
)
# Save as PNG
fig.write_image(
    "reports/figures/categorical_readmission_rates.png",
    width=1600,
    height=900,
    scale=2
)

# Display interactive figure
fig.show()


**Explanation / key finding:** Plots common categories with more than 200 observations and their confidence intervals. The display filters smaller groups, so it does not represent every category or constitute a fairness audit.

In [17]:
# Plot diag columns

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "Primary Diagnosis",
        "Secondary Diagnosis",
        "Additional Diagnosis"
    ],
    horizontal_spacing=0.08
)

for i, col in enumerate(diag_cols):

    # Get the 10 most common categories
    table = rate_table(col).head(10).reset_index()

    # Only show categories with more than 200 observations
    table = table[table["n"] > 200].copy()
    # Sort from highest to lowest readmission rate
    table = table.sort_values(
        "readmit_rate_pct",
        ascending=True
    ).reset_index(drop=True)

    # Calculate 95% Wilson confidence intervals
    ci = table.apply(
        lambda row: wilson_ci(
            row["positives"],
            row["n"]
        ),
        axis=1
    )

    table["ci_low"] = [x[0] for x in ci]
    table["ci_high"] = [x[1] for x in ci]

    # Error bar lengths
    table["error_low"] = (
        table["readmit_rate_pct"] - table["ci_low"]
    )

    table["error_high"] = (
        table["ci_high"] - table["readmit_rate_pct"]
    )

    # Highlight categories above base rate
    colors = [
        "#D55E00" if rate > base_rate else "#BDBDBD"
        for rate in table["readmit_rate_pct"]
    ]

    # Add bar chart
    fig.add_trace(
        go.Bar(
            x=table[col].astype(str),
            y=table["readmit_rate_pct"],
            marker_color=colors,

            error_y=dict(
                type="data",
                symmetric=False,
                array=table["error_high"],
                arrayminus=table["error_low"],
                visible=True
            ),

            customdata=np.stack(
                (
                    table["n"],
                    table["ci_low"],
                    table["ci_high"]
                ),
                axis=-1
            ),

            hovertemplate=(
                "Diagnosis: %{x}<br>"
                "N: %{customdata[0]:,.0f}<br>"
                "Readmission rate: %{y:.1f}%<br>"
                "95% CI: %{customdata[1]:.1f}%–%{customdata[2]:.1f}%"
                "<extra></extra>"
            ),

            showlegend=False
        ),
        row=1,
        col=i + 1
    )

    # Overall base rate
    fig.add_hline(
        y=base_rate,
        line_dash="dash",
        line_color="black",
        line_width=1,
        row=1,
        col=i + 1
    )

    fig.update_yaxes(
        title_text="Readmission rate (%)",
        row=1,
        col=i + 1
    )

    fig.update_xaxes(
        tickangle=-45,
        row=1,
        col=i + 1
    )


fig.update_layout(
    title=(
        "30-Day Readmission Rate by Diagnosis Group"
        f"<br><sup>Dashed line = overall training base rate ({base_rate:.1f}%) | "
        "Highlighted bars = above base rate | Error bars = 95% CI</sup>"
    ),
    template="plotly_white",
    height=600,
    width=1400
)

# Save as PNG
fig.write_image(
    "reports/figures/diagnosis_readmission_rates.png",
    width=1400,
    height=600,
    scale=2
)

fig.show()


**Explanation / key finding:** Compares readmission rates across primary, secondary, and additional diagnosis groups. The plots describe associations within training data; diagnosis position and group size should inform interpretation.

### IDS Mapping

In [18]:
import pandas as pd

id_mapping = pd.read_csv(
    "data/raw/IDS_mapping.csv",
    header=None,
    names=["id", "description"]
)


**Explanation / key finding:** Reads the ID mapping file for category descriptions. The file contains several mapping sections, so the next parser builds a separate dictionary for each administrative variable.

In [19]:
import csv

id_maps = {}

current_col = None

id_cols = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

with open("data/raw/IDS_mapping.csv", encoding="utf-8") as file:

    for row in csv.reader(file):

        if not row:
            continue

        value = row[0].strip()

        # Start of a new mapping table
        if value in id_cols:
            current_col = value
            id_maps[current_col] = {}

        # Add ID and description
        elif current_col and len(row) > 1:
            id_maps[current_col][value] = row[1].strip()


**Explanation / key finding:** Parses separate admission, discharge, and source mappings. Human-readable labels make plots interpretable without changing model predictors.

In [20]:
# Show readable categories

cols_to_show = [
    "discharge_disposition_id",
    "admission_source_id",
    "admission_type_id"
]

for col in cols_to_show:

    table = rate_table(col).reset_index()

    # Keep categories with at least 200 patients
    table = table[table["n"] >= 200].copy()

    # Add readable description
    table["description"] = (
        table[col].map(id_maps[col])
    )

    # Sort highest readmission rate first
    table = table.sort_values(
        "readmit_rate_pct",
        ascending=False
    )

    print(f"\n=== {col} ===")

    print(
        table[
            [col, "description", "n", "positives", "readmit_rate_pct"]
        ].to_string(index=False)
    )



=== discharge_disposition_id ===
discharge_disposition_id                                                                       description     n  positives  readmit_rate_pct
                      22 Discharged/transferred to another rehab fac including rehab units of a hospital .   986        267             27.08
                       5              Discharged/transferred to another type of inpatient care institution   650        133             20.46
                       3                                                     Discharged/transferred to SNF  6175        853             13.81
                       2                             Discharged/transferred to another short term hospital  1076        140             13.01
                      18                                                                              NULL  1701        175             10.29
                       4                                                     Discharged/transferred to ICF   364  

**Explanation / key finding:** Displays administrative category rates with descriptions and a minimum sample-size filter. Discharge differences may motivate predictors, but do not establish a causal effect of destination.

In [21]:

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "Discharge Disposition",
        "Admission Source",
        "Admission Type"
    ],
    horizontal_spacing=0.12
)

for i, col in enumerate(cols_to_show):

    # Get category summary
    table = rate_table(col).reset_index()

    # Only show categories with more than 200 observations
    table = table[table["n"] > 200].copy()

    # Add readable description
    table["description"] = table[col].map(id_maps[col])

    # Create readable label
    table["label"] = (
        table[col].astype(str)
        + ": "
        + table["description"].fillna("Unknown")
    )

    # Sort lowest to highest so highest appears at the top
    table = table.sort_values(
        "readmit_rate_pct",
        ascending=True
    ).reset_index(drop=True)

    # Calculate 95% Wilson confidence intervals
    ci = table.apply(
        lambda row: wilson_ci(
            row["positives"],
            row["n"]
        ),
        axis=1
    )

    table["ci_low"] = [x[0] for x in ci]
    table["ci_high"] = [x[1] for x in ci]

    # Error bar lengths
    table["error_low"] = (
        table["readmit_rate_pct"] - table["ci_low"]
    )

    table["error_high"] = (
        table["ci_high"] - table["readmit_rate_pct"]
    )

    # Highlight categories above base rate
    colors = [
        "#D55E00" if rate > base_rate else "#BDBDBD"
        for rate in table["readmit_rate_pct"]
    ]

    # Add horizontal bar chart
    fig.add_trace(
        go.Bar(
            x=table["readmit_rate_pct"],
            y=table["label"],
            orientation="h",
            marker_color=colors,

            error_x=dict(
                type="data",
                symmetric=False,
                array=table["error_high"],
                arrayminus=table["error_low"],
                visible=True
            ),

            customdata=np.stack(
                (
                    table["n"],
                    table["ci_low"],
                    table["ci_high"]
                ),
                axis=-1
            ),

            hovertemplate=(
                "%{y}<br>"
                "N: %{customdata[0]:,.0f}<br>"
                "Readmission rate: %{x:.1f}%<br>"
                "95% CI: %{customdata[1]:.1f}%–%{customdata[2]:.1f}%"
                "<extra></extra>"
            ),

            showlegend=False
        ),
        row=1,
        col=i + 1
    )

    # Overall base rate
    fig.add_vline(
        x=base_rate,
        line_dash="dash",
        line_color="black",
        line_width=1,
        row=1,
        col=i + 1
    )

    fig.update_xaxes(
        title_text="Readmission rate (%)",
        row=1,
        col=i + 1
    )

    fig.update_layout(
    title=(
        "30-Day Readmission Rate by Admission and Discharge Categories"
        f"<br><sup>n > 200 | Dashed line = overall base rate ({base_rate:.1f}%) | "
        "Highlighted bars = above base rate | Error bars = 95% CI</sup>"
    ),
    template="plotly_white",
    height=750,
    width=1800
)

fig.write_image(
    "reports/figures/admission_discharge_readmission_rates.png",
    width=1800,
    height=750,
    scale=2
)

fig.show()


**Explanation / key finding:** Plots administrative-category rates relative to the training base rate. Highlighting an above-average group identifies an observed association, not a model explanation or a proven intervention target.

# Findings

Class imbalance: 8.97% of patient were readmitted within 30 days, or roughly 1 in 11. So a model that would always predict "not readmiited" would be 91% correct most of the time and therefore useless. A corect measure would be the ROC-AUC, PRC-AUC and recall not accuracy

Age: Readmission increases with age
Gender: 8.99% vs 8.95% means no difference
Race: the rates are close with 9.18% for Caucasian and 8.65% for African American.
A1C: 8-9% across categories, so it does not separate patients
Glucose: >200 has the highest at 11.7% but they only comprise 658 patients; majority were not tested

Discharge disposition shows one of the largest differences in observed 30-day readmission rates among the categorical variables examined. For patient discharged at home, which is the majority, 6.8% were readmitted but for those who were transferred to rehab, 27.1% were readmitted. 
I would like to predict if after discharge, the patient will be likely readmitted within 30 days
